In [1]:
# Importar as bibliotecas

import pandas as pd
import re

from datetime import time
from pyspark.sql.functions import current_timestamp


StatementMeta(, 67a5cc47-c611-410a-af53-0d2ce39d2bc4, 3, Finished, Available, Finished)

In [2]:
# Ler URL do Excel

url = (
    "https://teste7857118138.blob.core.windows.net/azureml/"
    "Base%20SAC.xlsx?"
    "sp=r&st=2026-02-07T16:15:43Z"
    "&se=2026-02-08T00:30:43Z"
    "&spr=https"
    "&sv=2024-11-04"
    "&sr=b"
    "&sig=H3YESyvwvLZ8TTgFIH%2BrWFRuuhbCN%2BgC2TqTQAsoT3E%3D"
)

df = pd.read_excel(
    url,
    sheet_name="dUsuario"
)



StatementMeta(, 67a5cc47-c611-410a-af53-0d2ce39d2bc4, 4, Finished, Available, Finished)

In [3]:
#Converter campos datetime.time para string

for col in df.columns:
    df[col] = df[col].apply(
        lambda x: x.strftime('%H:%M:%S')
        if isinstance(x, time)
        else x
    )

StatementMeta(, 67a5cc47-c611-410a-af53-0d2ce39d2bc4, 5, Finished, Available, Finished)

In [4]:
#Normalizar nomes das colunas

def normalize_col(col):
    col = col.strip().lower()
    col = re.sub(r"[^\w]", "_", col)
    col = re.sub(r"_+", "_", col)
    return col

df.columns = [normalize_col(c) for c in df.columns]

StatementMeta(, 67a5cc47-c611-410a-af53-0d2ce39d2bc4, 6, Finished, Available, Finished)

In [5]:
#Converter Pandas → Spark

df_spark = spark.createDataFrame(df)

StatementMeta(, 67a5cc47-c611-410a-af53-0d2ce39d2bc4, 7, Finished, Available, Finished)

In [6]:
#Adicionar coluna de auditoria

df_spark = df_spark.withColumn(
    "data_atualizacao",
    current_timestamp()
)

StatementMeta(, 67a5cc47-c611-410a-af53-0d2ce39d2bc4, 8, Finished, Available, Finished)

In [8]:
#Criar schema se não existir   Definir a tabela Dimensão

spark.sql("CREATE SCHEMA IF NOT EXISTS dbo")

tabela = "dbo.dim_usuario"


if not spark.catalog.tableExists(tabela):
    (
        df_spark
        .limit(0)
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tabela)
    )

StatementMeta(, 67a5cc47-c611-410a-af53-0d2ce39d2bc4, 10, Finished, Available, Finished)

In [9]:
#Ler dados existentes

df_dim = spark.table(tabela)

StatementMeta(, 67a5cc47-c611-410a-af53-0d2ce39d2bc4, 11, Finished, Available, Finished)

In [10]:
#Identificar novos usuários (incremental)

df_incremental = (
    df_spark.alias("n")
    .join(
        df_dim.alias("d"),
        on=["id_usuario"],
        how="left_anti"
    )
)

StatementMeta(, 67a5cc47-c611-410a-af53-0d2ce39d2bc4, 12, Finished, Available, Finished)

In [11]:
#Inserir novos registros na dimensão

(
    df_incremental.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(tabela)
)

StatementMeta(, 67a5cc47-c611-410a-af53-0d2ce39d2bc4, 13, Finished, Available, Finished)

In [12]:
#Validar a Dimensão

display(
    spark.sql("SELECT * FROM dbo.dim_usuario LIMIT 1000")
)

StatementMeta(, 67a5cc47-c611-410a-af53-0d2ce39d2bc4, 14, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 0b4ebd62-8a13-4720-be46-0bd23ce6babe)